In [1]:

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, r2_score, accuracy_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Earthquake Magnitude Prediction for Alert in Area
### Using Random Forest Regressor to predict earthquake magnitude.

In [2]:

df = pd.read_csv('Significant Earthquake Dataset 1900-2023.csv')
pd.set_option('display.max_columns', None)
df['Place'] = df['Place'].str.split(', ').str[-1]
df.head()


,Time,Place,Latitude,Longitude,Depth,Mag,MagType,nst,gap,dmin,rms,net,ID,Updated,Unnamed: 14,Type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2023-02-17T09:37:34.868Z,Indonesia,-6.5986,132.0763,38.615,6.1,mww,119.0,51.0,2.988,0.76,us,us6000jpl7,2023-02-17T17:58:24.040Z,NaN,earthquake,6.41,5.595,0.065,23.0,reviewed,us,us
1,2023-02-16T05:37:05.138Z,Vanuatu,-15.0912,167.0294,36.029,5.6,mww,81.0,26.0,0.392,0.94,us,us6000jpb1,2023-02-17T05:41:32.448Z,NaN,earthquake,5.99,6.080,0.073,18.0,reviewed,us,us
2,2023-02-15T18:10:10.060Z,Philippines,12.3238,123.8662,20.088,6.1,mww,148.0,47.0,5.487,0.54,us,us6000jp76,2023-02-16T20:12:32.595Z,NaN,earthquake,8.61,4.399,0.037,71.0,reviewed,us,us
3,2023-02-15T06:38:09.034Z,New Zealand,-40.5465,174.5709,74.320,5.7,mww,81.0,40.0,0.768,1.15,us,us6000jp1g,2023-02-16T06:42:09.738Z,NaN,earthquake,3.68,4.922,0.065,23.0,reviewed,us,us
4,2023-02-14T13:16:51.072Z,Romania,45.1126,23.1781,10.000,5.6,mww,132.0,28.0,1.197,0.40,us,us6000jnqz,2023-02-17T09:15:18.586Z,NaN,earthquake,4.85,1.794,0.032,95.0,reviewed,us,us


In [26]:

nan_rows_ = df['Place'].isna().sum()
nan_rows_

df.drop('Time', axis=1, inplace=True)
df.drop('MagType', axis=1, inplace=True)
df.drop('net', axis=1, inplace=True)
df.drop('ID', axis=1, inplace=True)

df['Updated'] = pd.to_datetime(df['Updated'])

df['Time_month'] = df['Updated'].dt.month
df['Time_day'] = df['Updated'].dt.day
df['Time_hour'] = df['Updated'].dt.hour

df = df.drop(['Updated'], axis=1)

df

,Place,Latitude,Longitude,Depth,Mag,nst,gap,dmin,rms,Unnamed: 14,Type,horizontalError,depthError,magError,magNst,status,locationSource,magSource,Time_month,Time_day,Time_hour
0,Indonesia,-6.5986,132.0763,38.615,6.10,119.0,51.0,2.988,0.76,NaN,earthquake,6.41,5.595,0.065,23.0,reviewed,us,us,2,17,17
1,Vanuatu,-15.0912,167.0294,36.029,5.60,81.0,26.0,0.392,0.94,NaN,earthquake,5.99,6.080,0.073,18.0,reviewed,us,us,2,17,5
2,Philippines,12.3238,123.8662,20.088,6.10,148.0,47.0,5.487,0.54,NaN,earthquake,8.61,4.399,0.037,71.0,reviewed,us,us,2,16,20
3,New Zealand,-40.5465,174.5709,74.320,5.70,81.0,40.0,0.768,1.15,NaN,earthquake,3.68,4.922,0.065,23.0,reviewed,us,us,2,16,6
4,Romania,45.1126,23.1781,10.000,5.60,132.0,28.0,1.197,0.40,NaN,earthquake,4.85,1.794,0.032,95.0,reviewed,us,us,2,17,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37326,Alaska,52.3800,-167.4500,NaN,7.00,NaN,NaN,NaN,NaN,NaN,earthquake,NaN,NaN,NaN,NaN,reviewed,ushis,abe,6,4,20
37327,Alaska,51.4500,-171.0200,NaN,7.10,NaN,NaN,NaN,NaN,NaN,earthquake,NaN,NaN,NaN,NaN,reviewed,ushis,abe,6,4,20
37328,south of Alaska,52.0000,-160.0000,NaN,7.00,NaN,NaN,NaN,NaN,NaN,earthquake,NaN,NaN,NaN,NaN,reviewed,ushis,abe,6,4,20
37329,California,36.0000,-120.5000,NaN,6.40,NaN,NaN,NaN,NaN,NaN,earthquake,NaN,NaN,NaN,NaN,reviewed,ushis,ell,6,4,20


In [27]:
nan_rows_ = df.isna().sum()
nan_rows_

columns_to_drop = ['nst', 'gap', 'dmin', 'rms', 'Unnamed: 14', 'Type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'magSource']
df.drop(columns=columns_to_drop, inplace=True)

df.dropna(subset=['Place', 'Depth'], inplace=True)

nan_rows_ = df.isna().sum()
nan_rows_

Place             0
Latitude          0
Longitude         0
Depth             0
Mag               0
locationSource    0
Time_month        0
Time_day          0
Time_hour         0
dtype: int64

In [28]:

df.head()
# df.drop(columns='status', inplace=True)
df.drop(columns='locationSource', inplace=True)
df.drop(columns='Place', inplace=True)
# df.drop(columns='magSource', inplace=True)
df.head()

,Latitude,Longitude,Depth,Mag,Time_month,Time_day,Time_hour
0,-6.5986,132.0763,38.615,6.1,2,17,17
1,-15.0912,167.0294,36.029,5.6,2,17,5
2,12.3238,123.8662,20.088,6.1,2,16,20
3,-40.5465,174.5709,74.320,5.7,2,16,6
4,45.1126,23.1781,10.000,5.6,2,17,9


In [29]:

X = df.drop('Mag', axis=1).values
y = df['Mag'].values
type(X), type(y)

(numpy.ndarray, numpy.ndarray)

In [30]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
ct = ColumnTransformer(transformers=[('encoder', OneHotEncoder(), [0])], remainder='passthrough')
X = np.array(ct.fit_transform(X))


In [31]:

from sklearn.ensemble import RandomForestRegressor
regressor = RandomForestRegressor(n_estimators=30, random_state=42)
regressor.fit(X_train, y_train)

rf_pred_x_train = regressor.predict(X_train)
rf_pred_x_test = regressor.predict(X_test)


# Metrics

In [32]:
rf_pred_x_train = regressor.predict(X_train)


In [33]:
rf_pred_x_test = regressor.predict(X_test)


In [34]:
r2_score(y_train, rf_pred_x_train)


0.8654416094337274

In [35]:
r2_score(y_test, rf_pred_x_test)


0.09394834299229293

In [38]:
mean_absolute_error(y_test, rf_pred_x_test)


0.3178072824958237

In [39]:
mean_squared_error(y_test, rf_pred_x_test)


0.19407690724420815

In [40]:
mean_squared_error(y_test, rf_pred_x_test, squared=False)


c:\Users\pubg3\.conda\envs\tf\lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


0.44054160671179304

# Evaluation:

### The model has good predictive performance, with high accuracy and reliable metrics.
### These confidence in the model's ability to effectively predict earthquake magnitudes.